# Mitsui GBM: Per-Exchange-Group LightGBM

Implements the plan in `hayden_plan.md`:
1. Feature engineering (lagged returns, rolling vol/zscore, spread features)
2. Regime features (copper/gold ratio, gold/platinum ratio, FX USD proxy, cross-sectional vol, calendar)
3. One LightGBM model per exchange group (8 groups), seed-averaged
4. Scored with `ddof=0` to match the official Kaggle scorer

In [1]:
import re
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import lightgbm as lgb

DATA_DIR     = '/Users/hayden/coderepos_mac_mini/mitsui_commodity/data'
HOLDOUT_START = 1827
LAGS         = [1, 2, 5, 10, 20]
VOL_WINDOWS  = [5, 10, 20]
N_SEEDS      = 3
VAL_FRAC     = 0.15   # fraction of training rows used for val / early stopping

train        = pd.read_csv(f'{DATA_DIR}/train.csv')
train_labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
target_pairs = pd.read_csv(f'{DATA_DIR}/target_pairs.csv')

print(f'train {train.shape}  labels {train_labels.shape}')

train (1961, 558)  labels (1961, 425)


In [2]:
# ── Parse utilities (from targets.ipynb) ──────────────────────────────────────

def parse_pair(pair_str):
    s = pair_str.strip()
    tokens = re.split(r'\s*([+-])\s*', s)
    result, sign = [], 1
    if tokens[0]:
        result.append((sign, tokens[0]))
    i = 1
    while i < len(tokens):
        op, ticker = tokens[i], tokens[i + 1]
        result.append((1 if op == '+' else -1, ticker))
        i += 2
    return result

def get_exchange(ticker):
    if ticker.startswith('US_Stock_'):
        return 'US_Stock'
    return ticker.split('_')[0]

def get_instrument(ticker):
    t = ticker
    for pfx in ('US_Stock_', 'LME_', 'JPX_', 'FX_'):
        if t.startswith(pfx):
            t = t[len(pfx):]
            break
    for sfx in ('_Close', '_adj_close', '_close'):
        if t.endswith(sfx):
            t = t[:-len(sfx)]
            break
    return t

In [3]:
# ── Build target_info ─────────────────────────────────────────────────────────

rows = []
for _, r in target_pairs.iterrows():
    components = parse_pair(r['pair'])
    tickers    = [t for _, t in components]
    exchanges  = sorted(set(get_exchange(t) for t in tickers))
    rows.append({
        'target':       r['target'],
        'lag':          r['lag'],
        'pair_raw':     r['pair'],
        'n_legs':       len(components),
        'is_spread':    len(components) > 1,
        'tickers':      tickers,
        'instruments':  [get_instrument(t) for t in tickers],
        'exchanges':    exchanges,
        'exchange_key': '_'.join(exchanges),
    })
target_info = pd.DataFrame(rows)

print(target_info['n_legs'].value_counts())
print(target_info.groupby('exchange_key').size().sort_values(ascending=False))

n_legs
2    420
1      4
Name: count, dtype: int64
exchange_key
LME_US_Stock    181
JPX_US_Stock     87
FX_LME           86
FX_JPX           46
JPX_LME          12
LME               8
FX                2
US_Stock          2
dtype: int64


In [4]:
# ── Feature engineering functions ─────────────────────────────────────────────

def build_engineered_features(df, cols, lags, vol_windows):
    """
    Returns (DataFrame, dict: eng_col → src_col).
    Engineers lagged returns, rolling vol, rolling z-score.
    """
    source_map = {}
    parts = []

    for c in cols:
        for k in lags:
            name = f'ret_{c}_lag{k}'
            parts.append(df[c].pct_change(k).rename(name))
            source_map[name] = c

    for c in cols:
        daily_ret = df[c].pct_change(1)
        for w in vol_windows:
            name = f'vol_{c}_w{w}'
            parts.append(daily_ret.rolling(w, min_periods=max(2, w // 2)).std().rename(name))
            source_map[name] = c

    for c in cols:
        for w in vol_windows:
            roll = df[c].rolling(w, min_periods=max(2, w // 2))
            name = f'zscore_{c}_w{w}'
            parts.append(((df[c] - roll.mean()) / (roll.std() + 1e-8)).rename(name))
            source_map[name] = c

    return pd.concat(parts, axis=1), source_map


def build_spread_features(df, target_info, raw_feature_cols):
    """Adds current spread + lag1 + lag5 for each unique 2-leg pair."""
    seen  = set()
    parts = []
    fc    = set(raw_feature_cols)
    for _, row in target_info.iterrows():
        if not row['is_spread'] or len(row['tickers']) != 2:
            continue
        t0, t1 = row['tickers']
        if t0 not in fc or t1 not in fc:
            continue
        key = (t0, t1)
        if key in seen:
            continue
        seen.add(key)
        spread = df[t0] - df[t1]
        base   = f'spr__{t0}__{t1}'
        parts.append(spread.rename(base))
        parts.append(spread.shift(1).rename(f'{base}_lag1'))
        parts.append(spread.shift(5).rename(f'{base}_lag5'))
    return pd.concat(parts, axis=1) if parts else pd.DataFrame(index=df.index)


def build_regime_features(df, raw_feature_cols):
    """Shared macro/calendar features appended to every group model."""
    fc  = set(raw_feature_cols)
    new = {}

    if 'LME_CA_Close' in fc and 'JPX_Gold_Standard_Futures_Close' in fc:
        new['regime_copper_gold'] = (
            df['LME_CA_Close'] / (df['JPX_Gold_Standard_Futures_Close'] + 1e-8)
        )

    if 'JPX_Gold_Standard_Futures_Close' in fc and 'JPX_Platinum_Standard_Futures_Close' in fc:
        new['regime_gold_platinum'] = (
            df['JPX_Gold_Standard_Futures_Close']
            / (df['JPX_Platinum_Standard_Futures_Close'] + 1e-8)
        )

    usd_pairs = [c for c in ('FX_EURUSD', 'FX_GBPUSD', 'FX_AUDUSD', 'FX_NZDUSD') if c in fc]
    if usd_pairs:
        new['regime_fx_usd_proxy'] = df[usd_pairs].mean(axis=1)

    us_adj = [c for c in raw_feature_cols if c.startswith('US_Stock_') and c.endswith('_adj_close')]
    if us_adj:
        rets = df[us_adj].pct_change(1)
        new['regime_xsec_vol'] = rets.rolling(20, min_periods=10).std().mean(axis=1)

    # Calendar proxies via trading-day integer modulo
    new['regime_dow'] = df['date_id'] % 5
    new['regime_dom'] = df['date_id'] % 21

    return pd.DataFrame(new, index=df.index)

In [5]:
# ── Build full feature matrix ──────────────────────────────────────────────────

raw_feature_cols = [c for c in train.columns if c != 'date_id']

# Engineer only from price-like columns to avoid bloat from volume/OI
price_cols = [
    c for c in raw_feature_cols
    if c.endswith('_adj_close') or c.endswith('_Close') or c.startswith('FX_')
]

print(f'{len(raw_feature_cols)} raw features, {len(price_cols)} price cols for engineering')

df = train.sort_values('date_id').reset_index(drop=True)

eng_df, source_map = build_engineered_features(df, price_cols, LAGS, VOL_WINDOWS)
spr_df             = build_spread_features(df, target_info, raw_feature_cols)
reg_df             = build_regime_features(df, raw_feature_cols)

full_df = pd.concat(
    [df[['date_id'] + raw_feature_cols], eng_df, spr_df, reg_df], axis=1
)

feat_cols = [c for c in full_df.columns if c != 'date_id']
full_df[feat_cols] = full_df[feat_cols].ffill().fillna(0)   # never bfill

print(f'full_df: {full_df.shape}')
print(f'  raw={len(raw_feature_cols)}, eng={eng_df.shape[1]}, '
      f'spreads={spr_df.shape[1]}, regime={reg_df.shape[1]}')

557 raw features, 143 price cols for engineering
full_df: (1961, 3247)
  raw=557, eng=1573, spreads=1110, regime=6


In [6]:
# ── Holdout split ─────────────────────────────────────────────────────────────

target_cols   = [c for c in train_labels.columns if c != 'date_id']
labels_sorted = train_labels.sort_values('date_id').reset_index(drop=True)

train_mask   = full_df['date_id'] < HOLDOUT_START
holdout_mask = full_df['date_id'] >= HOLDOUT_START

X_train_df = full_df[train_mask].reset_index(drop=True)
X_hold_df  = full_df[holdout_mask].reset_index(drop=True)

n_train   = len(X_train_df)
split_idx = int(n_train * (1 - VAL_FRAC))

# Training targets: fill NaN with 0 (required for LGBM)
y_train_all = (
    labels_sorted[labels_sorted['date_id'] < HOLDOUT_START]
    [target_cols].fillna(0).values.astype(np.float32)
)

# Holdout targets: preserve NaN for scoring mask
holdout_dates = X_hold_df['date_id'].values
y_hold_true   = (
    labels_sorted[labels_sorted['date_id'].isin(holdout_dates)]
    .sort_values('date_id')[target_cols].values
)

print(f'Train: {n_train} rows, val split at row {split_idx}, holdout: {len(X_hold_df)} rows')
print(f'y_train_all: {y_train_all.shape}, y_hold_true: {y_hold_true.shape}')

Train: 1827 rows, val split at row 1552, holdout: 134 rows
y_train_all: (1827, 424), y_hold_true: (134, 424)


In [7]:
# ── Scoring metric ────────────────────────────────────────────────────────────
# ddof=0 (population std) matches the official Kaggle scorer exactly.

def mitsui_metric(y_true, y_pred, verbose=False):
    daily_corrs = []
    for i in range(len(y_true)):
        mask = ~np.isnan(y_true[i])
        if mask.sum() < 2:
            continue
        corr, _ = spearmanr(y_true[i, mask], y_pred[i, mask])
        if not np.isnan(corr):
            daily_corrs.append(corr)
    arr = np.array(daily_corrs)
    std = arr.std(ddof=0)
    if std == 0:
        raise ZeroDivisionError('Daily corr std is zero.')
    sharpe = arr.mean() / std
    if verbose:
        print(f'  mean={arr.mean():.4f}  std={std:.4f}  sharpe={sharpe:.4f}  n_days={len(arr)}')
    return sharpe

In [8]:
# ── Group feature selection + GroupModel ──────────────────────────────────────

def get_group_feat_cols(exchange_key):
    """
    Feature columns for one exchange group:
      - raw OHLCV cols for exchanges in the group key
      - engineered cols (ret/vol/zscore) derived from those raw cols
      - spread cols for targets belonging to this group
      - all regime cols
    """
    exchanges = set(exchange_key.split('_'))

    raw = [c for c in raw_feature_cols if get_exchange(c) in exchanges]
    raw_set = set(raw)

    eng = [ec for ec, sc in source_map.items() if sc in raw_set]

    group_tickers = set(
        t
        for tl in target_info[target_info['exchange_key'] == exchange_key]['tickers']
        for t in tl
    )
    spr = [
        c for c in spr_df.columns
        if any(f'__{t}__' in c or c.endswith(f'__{t}') or c.endswith(f'__{t}_lag1') or c.endswith(f'__{t}_lag5')
               for t in group_tickers)
    ]

    reg = list(reg_df.columns)

    available = set(full_df.columns)
    return [c for c in sorted(set(raw) | set(eng) | set(spr) | set(reg)) if c in available]


class GroupModel:
    """One LGBMRegressor per target, early stopping, N seeds averaged."""

    def __init__(self, n_estimators=300, learning_rate=0.05, num_leaves=31,
                 min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                 seeds=(0, 1, 2)):
        self.base_params = dict(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            num_leaves=num_leaves,
            min_child_samples=min_child_samples,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            verbose=-1,
        )
        self.seeds = seeds
        self.models_per_seed = []
        self.target_names    = []

    def fit(self, X_tr, y_tr, X_val, y_val, target_names):
        self.target_names    = target_names
        self.models_per_seed = []
        for seed in self.seeds:
            params = {**self.base_params, 'random_state': seed}
            seed_models = []
            for i in range(len(target_names)):
                m = lgb.LGBMRegressor(**params)
                m.fit(
                    X_tr, y_tr[:, i],
                    eval_set=[(X_val, y_val[:, i])],
                    callbacks=[
                        lgb.early_stopping(30, verbose=False),
                        lgb.log_evaluation(period=0),
                    ],
                )
                seed_models.append(m)
            self.models_per_seed.append(seed_models)
            print(f'  seed {seed} done')

    def predict(self, X):
        preds = np.zeros((len(X), len(self.target_names)))
        for seed_models in self.models_per_seed:
            preds += np.column_stack([m.predict(X) for m in seed_models])
        return preds / len(self.models_per_seed)

In [9]:
# ── Train per-exchange-group models ───────────────────────────────────────────

exchange_keys = sorted(target_info['exchange_key'].unique())
group_models  = {}
group_fcols   = {}

for ekey in exchange_keys:
    group_targets = target_info[target_info['exchange_key'] == ekey]['target'].tolist()
    fcols         = get_group_feat_cols(ekey)
    group_fcols[ekey] = fcols

    # Build X for training rows only
    X_full = X_train_df[fcols].values.astype(np.float32)
    y_idx  = [target_cols.index(t) for t in group_targets]
    y      = y_train_all[:, y_idx]

    X_tr  = X_full[:split_idx]
    y_tr  = y[:split_idx]
    X_val = X_full[split_idx:]
    y_val = y[split_idx:]

    print(f'\n=== {ekey} | {len(group_targets)} targets | {len(fcols)} features ===')
    gm = GroupModel(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        seeds=(0, 1, 2),
    )
    gm.fit(X_tr, y_tr, X_val, y_val, group_targets)
    group_models[ekey] = gm

print('\nAll groups trained.')


=== FX | 2 targets | 486 features ===
  seed 0 done
  seed 1 done
  seed 2 done

=== FX_JPX | 46 targets | 1123 features ===
  seed 0 done
  seed 1 done
  seed 2 done

=== FX_LME | 86 targets | 1377 features ===
  seed 0 done
  seed 1 done
  seed 2 done

=== JPX_LME | 12 targets | 1270 features ===
  seed 0 done
  seed 1 done
  seed 2 done

=== JPX_US_Stock | 87 targets | 838 features ===
  seed 0 done
  seed 1 done
  seed 2 done

=== LME | 8 targets | 804 features ===
  seed 0 done
  seed 1 done
  seed 2 done

=== LME_US_Stock | 181 targets | 1020 features ===
  seed 0 done
  seed 1 done
  seed 2 done

=== US_Stock | 2 targets | 27 features ===
  seed 0 done
  seed 1 done
  seed 2 done

All groups trained.


In [10]:
# ── Predict holdout ───────────────────────────────────────────────────────────

predictions = np.zeros((len(X_hold_df), len(target_cols)))

for ekey, gm in group_models.items():
    fcols      = group_fcols[ekey]
    X_hold     = X_hold_df[fcols].values.astype(np.float32)
    preds      = gm.predict(X_hold)
    y_idx      = [target_cols.index(t) for t in gm.target_names]
    predictions[:, y_idx] = preds

print(f'predictions: {predictions.shape},  holdout truth: {y_hold_true.shape}')

/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/pyth

predictions: (134, 424),  holdout truth: (134, 424)


/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/hayden/coderepos_mac_mini/mitsui_commodity/.venv/lib/pyth

In [11]:
# ── Score ─────────────────────────────────────────────────────────────────────

print('=== Per-group Sharpe ===')
for ekey, gm in group_models.items():
    y_idx    = [target_cols.index(t) for t in gm.target_names]
    g_true   = y_hold_true[:, y_idx]
    g_pred   = predictions[:, y_idx]
    try:
        s = mitsui_metric(g_true, g_pred)
        print(f'  {ekey:<20} {s:.4f}')
    except ZeroDivisionError as e:
        print(f'  {ekey:<20} ERROR: {e}')

print('\n=== Overall Sharpe (all 424 targets) ===')
score = mitsui_metric(y_hold_true, predictions, verbose=True)
print(f'\nFinal holdout Sharpe: {score:.4f}')

=== Per-group Sharpe ===
  FX                   0.1820
  FX_JPX               -0.0350
  FX_LME               0.1753
  JPX_LME              -0.1459
  JPX_US_Stock         0.0613
  LME                  0.0824
  LME_US_Stock         0.1750
  US_Stock             -0.0339

=== Overall Sharpe (all 424 targets) ===
  mean=0.0214  std=0.1304  sharpe=0.1644  n_days=134

Final holdout Sharpe: 0.1644
